# Task 11 — Surveillance / Video / Live

**Dataset:** Synthetic tracking video smoke

**Protocol:** smoke

**Models:** ByteTrack, OC-SORT, OSNet (expected_blocker if not installed), video annotate

In [ ]:
import json, sys, os
from pathlib import Path

# Add shared dir to path
NB_ROOT = Path(__file__).parent.parent if "__file__" in dir() else Path("/home/arash/PycharmProjects/VisionServeX/notebook")
sys.path.insert(0, str(NB_ROOT / "shared"))
os.chdir(str(NB_ROOT.parent))

from paths import COCO_400_ANN, COCO_400_IMAGES, SMOKE_IMG, SMOKE_ANN, NB_ROOT, REPO_ROOT
from display import clean, scan_text
from commands import run
from notebook_utils import write_status

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

import visionservex
print(f"VisionServeX {visionservex.__version__}")

TASK = NB_ROOT / '11_surveillance_video_live'


In [ ]:
from pathlib import Path
video = str(NB_ROOT / ".." / "tests/assets/smoke/tracking_sample.mp4")
person_img = str(NB_ROOT / ".." / "tests/assets/smoke/coco_person_car.jpg")
rows = []
# Tracker doctor
for tracker in ["bytetrack","oc-sort"]:
    r = run(["video-search","tracker-smoke","--tracker",tracker,
             "--format","json","--out",str(TASK/f"reports/{tracker}_smoke.json")], timeout=30)
    rows.append({"component":f"tracker:{tracker}",
                 "final_state":"smoke_passed" if r.get("status")=="ok" else "expected_blocker",
                 "code":r.get("code","-")})
# ReID doctor
r = run(["video-search","reid-smoke","--image",person_img,"--reid","osnet",
         "--format","json","--out",str(TASK/"reports/osnet_smoke.json")], timeout=30)
rows.append({"component":"reid:osnet","final_state":"smoke_passed" if r.get("status")=="ok" else "expected_blocker",
             "code":r.get("code","-")})
# Video annotate
r = run(["annotate","video","--video",video,"--model","dfine-s-o365-coco",
         "--out",str(TASK/"visuals/video_annotated.mp4")], timeout=180)
rows.append({"component":"video_annotate","final_state":"smoke_passed" if r.get("_returncode")==0 else "expected_blocker","code":"-"})
# Live dry-run
r = run(["live","--source","0","--model","dfine-s-o365-coco","--dry-run",
         "--out",str(TASK/"reports/live_dry_run.json")], timeout=30)
rows.append({"component":"live_dry_run","final_state":"smoke_passed" if r.get("_returncode")==0 else "expected_blocker","code":"-"})
df = pd.DataFrame(rows)
df.to_csv(TASK/"reports/surveillance_status.csv", index=False)
print(df.to_string(index=False))


In [ ]:
write_status(TASK, task='11_surveillance_video_live', dataset='Synthetic tracking video smoke', protocol='smoke')
print('Status written.')